In [1]:

import rioxarray as rxr
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
file_Chinon = "GHS_BUILT_C_MSZ_Chinon_32630.tif"
file_Ohio = "GHS_BUILT_C_MSZ_Ohio_32616_cut.tif"
file_greece = "GHS_BUILT_C_MSZ_Greece_32634_cut.tif"
file_PortoAlegre = "GHS_BUILT_C_MSZ_PortoAlegre_32722_cut.tif"


In [ ]:
rxr_Chinon = rxr.open_rasterio(file_Chinon)
rxr_Ohio = rxr.open_rasterio(file_Ohio)
rxr_greece = rxr.open_rasterio(file_greece)
rxr_PortoAlegre = rxr.open_rasterio(file_PortoAlegre)

rxr_Chinon = rxr_Chinon.rio.write_nodata(255)
rxr_Ohio = rxr_Ohio.rio.write_nodata(255)
rxr_greece = rxr_greece.rio.write_nodata(255)
rxr_PortoAlegre = rxr_PortoAlegre.rio.write_nodata(255)

In [4]:
class_urban_file = "GHS_BUILT_C_MSZ_color.clr"
with open(class_urban_file, "r") as f:
    class_urban = f.readlines()
    class_urban = [x.strip() for x in class_urban]
    class_urban = [x.split(', ') for x in class_urban]
    color_urban = [x[0].split(' ') for x in class_urban]
    for x in class_urban:
        x[0] = x[0].split(' ')[0]
        if len(x) == 3:
            x.insert(1, x[1])
    class_urban = np.array(class_urban)
    class_urban[:,0] = [x[0] for x in color_urban]
    color_urban = np.array([[int(x[1]), int(x[2]), int(x[3])] for x in color_urban])
    
class_urban = np.concatenate((class_urban, color_urban), axis=1)

pd_urban = pd.DataFrame(class_urban, columns=["raster_index", "Spatial domain", "Built-up type", "Classification", "R", "G", "B"])
pd_urban["simpler classification"] = pd_urban["Classification"]
pd_urban

# raster_index in 1,2,3,4 then simpler classification = "open space"
pd_urban.loc[pd_urban["raster_index"].isin(["1", "2", "3", "4", "5"]), "simpler classification"] = "open space"

In [ ]:
# count the number of pixels for each class
pd_urban["count_Chinon"] = 0
pd_urban["count_Ohio"] = 0
pd_urban["count_greece"] = 0
pd_urban["count_PortoAlegre"] = 0

for iindex in pd_urban["raster_index"]:
    pd_urban.loc[pd_urban["raster_index"] == iindex, "count_Chinon"] = rxr_Chinon.where(rxr_Chinon == int(iindex)).count(dim=['x','y']).values[0]
    pd_urban.loc[pd_urban["raster_index"] == iindex, "count_Ohio"] = rxr_Ohio.where(rxr_Ohio == int(iindex)).count(dim=['x','y']).values[0]
    pd_urban.loc[pd_urban["raster_index"] == iindex, "count_greece"] = rxr_greece.where(rxr_greece == int(iindex)).count(dim=['x','y']).values[0]
    pd_urban.loc[pd_urban["raster_index"] == iindex, "count_PortoAlegre"] = rxr_PortoAlegre.where(rxr_PortoAlegre == int(iindex)).count(dim=['x','y']).values[0]
    
total_Chinon = rxr_Chinon.where(rxr_Chinon.isin(pd_urban["raster_index"].astype(int))).count(dim=['x','y']).values[0]
total_Ohio = rxr_Ohio.where(rxr_Ohio.isin(pd_urban["raster_index"].astype(int))).count(dim=['x','y']).values[0]
total_greece = rxr_greece.where(rxr_greece.isin(pd_urban["raster_index"].astype(int))).count(dim=['x','y']).values[0]
total_PortoAlegre = rxr_PortoAlegre.where(rxr_PortoAlegre.isin(pd_urban["raster_index"].astype(int))).count(dim=['x','y']).values[0]

# set percentage of each class
pd_urban["percentage_Chinon"] =      100*(pd_urban["count_Chinon"] / total_Chinon)
pd_urban["percentage_Ohio"] =        100*(pd_urban["count_Ohio"] / total_Ohio)
pd_urban["percentage_greece"] =      100*(pd_urban["count_greece"] / total_greece)
pd_urban["percentage_PortoAlegre"] = 100*(pd_urban["count_PortoAlegre"] / total_PortoAlegre)

# replace "building height <=3m" by "building height <= 3m" to avoid problems with the latex table
pd_urban.loc[pd_urban["simpler classification"] == "building height <=3m", "simpler classification"] = "building height <= 3m"

pd_simpler = pd_urban.groupby("simpler classification").sum()
pd_simpler["percentage_Chinon"] =      100*(pd_simpler["count_Chinon"] / total_Chinon)
pd_simpler["percentage_Ohio"] =        100*(pd_simpler["count_Ohio"] / total_Ohio)
pd_simpler["percentage_greece"] =      100*(pd_simpler["count_greece"] / total_greece)
pd_simpler["percentage_PortoAlegre"] = 100*(pd_simpler["count_PortoAlegre"] / total_PortoAlegre)

try:
    pd_simpler.pop("R")
    pd_simpler.pop("G")
    pd_simpler.pop("B")
    pd_simpler.pop("raster_index")
    pd_simpler.pop("Spatial domain")
    pd_simpler.pop("Built-up type")
    pd_simpler.pop("Classification")
    pd_simpler.pop("count_Chinon")
    pd_simpler.pop("count_Ohio")
    pd_simpler.pop("count_greece")
    pd_simpler.pop("count_PortoAlegre")
except:
    pass

# rename columns
pd_simpler = pd_simpler.rename(columns={"percentage_Chinon": "Chinon", "percentage_Ohio": "Ohio", "percentage_greece": "Greece", "percentage_PortoAlegre": "Porto Alegre"})
# rename "simpler classification":"Urban Classification"
pd_simpler = pd_simpler.rename_axis("Urban Classification")

pd_simpler.to_latex("urban_classification.tex", float_format="%.2f", bold_rows=True, )
pd_simpler

total_greece_flooded 1388778
total_PortoAlegre_flooded 1065358


In [32]:
flooded_greece = gpd.read_file("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/EMSR_692/aux_data/EMSR692_aois_V2.gpkg")
flooded_PortoAlegre = gpd.read_file("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/PortoAlegre/aux_data/20240506_TotalFlood_delineation_dissolved.gpkg")

flooded_greece = flooded_greece.to_crs("EPSG:32634")
flooded_PortoAlegre = flooded_PortoAlegre.to_crs("EPSG:32722")

rxr_Greece_flooded = rxr_greece.rio.clip(flooded_greece.geometry)
rxr_PortoAlegre_flooded = rxr_PortoAlegre.rio.clip(flooded_PortoAlegre.geometry)

pd_urban_flooded = pd_urban.copy()
pd_urban_flooded["count_greece_flooded"] = 0
pd_urban_flooded["percentage_greece_flooded"] = 0
pd_urban_flooded["count_PortoAlegre_flooded"] = 0
pd_urban_flooded["percentage_PortoAlegre_flooded"] = 0

for iindex in pd_urban_flooded["raster_index"]:
    pd_urban_flooded.loc[pd_urban_flooded["raster_index"] == iindex, "count_greece_flooded"] = rxr_Greece_flooded.where(rxr_Greece_flooded == int(iindex)).count(dim=['x','y']).values[0]
    pd_urban_flooded.loc[pd_urban_flooded["raster_index"] == iindex, "count_PortoAlegre_flooded"] = rxr_PortoAlegre_flooded.where(rxr_PortoAlegre_flooded == int(iindex)).count(dim=['x','y']).values[0]
    
total_greece_flooded = rxr_Greece_flooded.where(rxr_Greece_flooded.isin(pd_urban_flooded["raster_index"].astype(int))).count(dim=['x','y']).values[0]
total_PortoAlegre_flooded = rxr_PortoAlegre_flooded.where(rxr_PortoAlegre_flooded.isin(pd_urban_flooded["raster_index"].astype(int))).count(dim=['x','y']).values[0]
print("total_greece_flooded", total_greece_flooded)
print("total_PortoAlegre_flooded", total_PortoAlegre_flooded)

pd_urban_flooded["percentage_greece_flooded"] = 100*(pd_urban_flooded["count_greece_flooded"] / total_greece_flooded)
pd_urban_flooded["percentage_PortoAlegre_flooded"] = 100*(pd_urban_flooded["count_PortoAlegre_flooded"] / total_PortoAlegre_flooded)

pd_urban_flooded.loc[pd_urban_flooded["simpler classification"] == "building height <=3m", "simpler classification"] = "building height <= 3m"
pd_simpler_flooded = pd_urban_flooded.groupby("simpler classification").sum()
pd_simpler_flooded["percentage_greece_flooded"] = 100*(pd_simpler_flooded["count_greece_flooded"] / total_greece_flooded)
pd_simpler_flooded["percentage_PortoAlegre_flooded"] = 100*(pd_simpler_flooded["count_PortoAlegre_flooded"] / total_PortoAlegre_flooded)
# pd_simpler_flooded

try:
    pd_simpler_flooded.pop("R")
    pd_simpler_flooded.pop("G")
    pd_simpler_flooded.pop("B")
    pd_simpler_flooded.pop("raster_index")
    pd_simpler_flooded.pop("Spatial domain")
    pd_simpler_flooded.pop("Built-up type")
    pd_simpler_flooded.pop("Classification")
    pd_simpler_flooded.pop("count_Chinon")
    pd_simpler_flooded.pop("count_Ohio")
    pd_simpler_flooded.pop("count_greece")
    pd_simpler_flooded.pop("count_PortoAlegre")
    pd_simpler_flooded.pop("count_greece_flooded")
    pd_simpler_flooded.pop("count_PortoAlegre_flooded")
except:
    pass

# rename columns
pd_simpler_flooded = pd_simpler_flooded.rename(columns={"percentage_Chinon": "Chinon", "percentage_Ohio": "Ohio", "percentage_greece": "Greece", "percentage_PortoAlegre": "Porto Alegre", "percentage_greece_flooded": "Greece within flooded areas", "percentage_PortoAlegre_flooded": "Porto Alegre within flooded area"})
# rename "simpler classification":"Urban Classification"
pd_simpler_flooded = pd_simpler_flooded.rename_axis("Urban Classification")

pd_simpler_flooded.to_latex("urban_classification_with_flood.tex", float_format="%.2f", bold_rows=True, )
pd_simpler_flooded

total_greece_flooded 1388778
total_PortoAlegre_flooded 1065358


,Chinon,Ohio,Greece,Porto Alegre,Greece within flooded areas,Porto Alegre within flooded area
Urban Classification,,,,,,
15m < building height <= 30m,0.338244,1.285298,0.587027,3.389349,0.700400,2.045510
3m < building height <= 6m,10.033293,8.430543,5.661819,3.569992,5.191182,4.445736
6m < building height <= 15m,6.105793,13.331988,8.106608,32.469486,6.880077,36.097537
building height <= 3m,31.068224,24.968968,28.997269,9.358480,30.430350,8.444955
building height > 30m,0.435369,0.172273,0.000000,0.240310,0.000000,0.167737
open space,52.019077,51.810930,56.647278,50.972382,56.797991,48.798526
